# Notebook 03 - Transform Operational Data to Silver

## Objective

Transform the Bronze layer into the Silver layer by resolving the data quality issues identified during data profiling.

The Silver layer contains clean, standardized, and validated data that can be trusted for downstream analytics.

## Pipeline Position
This is **Notebook 03** in the Medallion pipeline. It reads raw `bronze_*` Delta tables (generated in Notebook 01), applies data cleansing, deduplication, null filling, and schema casting rules informed by Notebook 02 profiling findings, and persists cleansed `silver_*` Delta tables for downstream Gold analytical modeling.

## Transformations

- Remove duplicate records
- Handle missing values
- Standardize inconsistent values
- Validate data types
- Create Silver Delta tables

## Output

Silver Delta Tables

- silver_customers
- silver_products
- silver_orders
- silver_order_lines
- silver_campaign_performance
- silver_sales_targets

# Section 1 - Customer Data Cleansing

## Objective
Deduplicate customer records on `CustomerID`, fill missing email fields with a default domain address (`unknown@verdanova.com`), standardize region casing with `initcap`, and parse `JoinDate` into proper date type for `silver_customers`.

In [1]:
from pyspark.sql.functions import *

customers = spark.table("bronze_customers")

customers = customers.dropDuplicates(["CustomerID"])

customers = customers.fillna({
    "Email": "unknown@verdanova.com"
})

customers = customers.withColumn(
    "Region",
    initcap(col("Region"))
)

customers = customers.withColumn(
    "JoinDate",
    to_date(col("JoinDate"))
)

customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_customers")

display(spark.table("silver_customers"))

StatementMeta(, 30c26902-3771-4c4e-8013-8ad30fbda65f, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7580d19f-b9c3-42b5-b990-858a31ff9a70)

# Section 2 - Product Data Cleansing

## Objective
Cleanse raw product data by deduplicating on `ProductID`, standardizing text formatting across categories, and validating numerical pricing attributes for `silver_products`.

In [3]:
from pyspark.sql.functions import *

products = spark.table("bronze_products")

products = products.dropDuplicates(["ProductID"])

products = products.withColumn(
    "Category",
    initcap(col("Category"))
)

# Fix intentional data-quality issue
products = products.withColumn(
    "Category",
    when(
        col("ProductID") == 9,
        lit("Cleaning")
    ).otherwise(col("Category"))
)

products = products.withColumn(
    "UnitPrice",
    col("UnitPrice").cast("decimal(18,2)")
)

products = products.withColumn(
    "StandardCost",
    col("StandardCost").cast("decimal(18,2)")
)

products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_products")

display(spark.table("silver_products"))

StatementMeta(, 7b835115-fe1a-4da8-827c-f7595b17f222, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, afcbfb52-a66f-419b-bae6-e697001876fd)

# Section 3 - Orders & Order Lines Cleansing

## Objective
Parse transaction dates in `bronze_orders` and enforce schema types (integers, explicit decimals) and key uniqueness on line items in `bronze_order_lines` prior to writing `silver_orders` and `silver_order_lines`.

In [1]:
from pyspark.sql.functions import *

orders = spark.table("bronze_orders")

orders = orders.withColumn(
    "OrderDate",
    to_date(col("OrderDate"))
)

orders.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_orders")

display(spark.table("silver_orders"))

StatementMeta(, b6165c34-55da-4c07-b448-1023e9cd5f1b, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 09e47472-420f-4dd6-b6a9-f562b2cdfd5a)

**Order Lines**

In [2]:
from pyspark.sql.functions import *

order_lines = spark.table("bronze_order_lines")

order_lines = order_lines.dropDuplicates(["OrderLineID"])

order_lines = order_lines.select(
    "OrderLineID",
    "OrderID",
    "ProductID",
    "Quantity",
    "UnitPrice",
    "CostAtSale",
    "Discount",
    "LineAmount"
)

order_lines = order_lines \
    .withColumn("Quantity", col("Quantity").cast("int")) \
    .withColumn("UnitPrice", col("UnitPrice").cast("decimal(18,2)")) \
    .withColumn("CostAtSale", col("CostAtSale").cast("decimal(18,2)")) \
    .withColumn("Discount", col("Discount").cast("decimal(18,2)")) \
    .withColumn("LineAmount", col("LineAmount").cast("decimal(18,2)"))

order_lines.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_order_lines")

display(spark.table("silver_order_lines"))

StatementMeta(, b6165c34-55da-4c07-b448-1023e9cd5f1b, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7fe72e20-1010-48f8-ad0e-75d069bab614)

# Section 4 - Marketing Campaign Cleansing

## Objective
Cleanse `bronze_campaign_performance` by deduplicating on `CampaignID`, imputing missing `Platform` values with default platform (`Instagram`), standardizing category casing, and casting financial metrics to `decimal(18,2)`.

In [1]:
from pyspark.sql.functions import *

campaigns = spark.table("bronze_campaign_performance")

campaigns = campaigns.dropDuplicates(["CampaignID"])

campaigns = campaigns.fillna({
    "Platform": "Instagram"
})

campaigns = campaigns.withColumn(
    "ProductCategory",
    initcap(col("ProductCategory"))
)

campaigns = campaigns \
    .withColumn("CampaignDate", to_date(col("CampaignDate"))) \
    .withColumn("Impressions", col("Impressions").cast("int")) \
    .withColumn("Clicks", col("Clicks").cast("int")) \
    .withColumn("Conversions", col("Conversions").cast("int")) \
    .withColumn("Spend", col("Spend").cast("decimal(18,2)")) \
    .withColumn("RevenueGenerated", col("RevenueGenerated").cast("decimal(18,2)"))

campaigns.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_campaign_performance")

display(spark.table("silver_campaign_performance"))

StatementMeta(, 46c77f72-91be-470b-951b-c8299728d1c5, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bc4b37a4-410e-48f6-9f25-d6b1980ca69f)

# Section 5 - Sales Targets Cleansing

## Objective
Standardize target planning data by imputing missing `SalesTarget` amounts with `0`, title-casing `Region` and `ProductCategory`, and casting date keys to explicit integer/decimal types for `silver_sales_targets`.

In [3]:
from pyspark.sql.functions import *

targets = spark.table("bronze_sales_targets")

targets = targets.fillna({
    "SalesTarget": 0
})

targets = targets.withColumn(
    "Region",
    initcap(col("Region"))
)

targets = targets.withColumn(
    "ProductCategory",
    initcap(col("ProductCategory"))
)

targets = targets \
    .withColumn("Year", col("Year").cast("int")) \
    .withColumn("Month", col("Month").cast("int")) \
    .withColumn("SalesTarget", col("SalesTarget").cast("decimal(18,2)"))

targets.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_sales_targets")

display(spark.table("silver_sales_targets"))

StatementMeta(, a9dd0e10-969a-4490-8efe-30aa0756a01d, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ffb3a2f-3741-4222-88a6-6287196dc393)

## Silver Layer Completed

The Silver layer now contains standardized and validated operational data.

### Completed Transformations

- Duplicate records removed
- Missing values handled
- Region values standardized
- Product categories standardized
- Silver Delta tables created

The Silver layer is now ready for dimensional modeling in the Gold layer.